In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

Session started : 2026-03-02 16:52
BQL service     : Service
NumPy 1.26.4  |  pandas 1.3.5


In [7]:
CONFIG_DIR = Path("config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg    = params["global"]
targets = params["performance_targets"]

# ── Canonical TSMOM parameters ───────────────────────────────────
def _get(cfg, key, default, label=""):
    if key in cfg:
        return cfg[key]
    print(f"  Warning: {label or key} missing from config, using default {default}")
    return default

LOOKBACK_DAYS = 252
TARGET_VOL    = _get(gcfg, "target_portfolio_vol_annual", 0.10, "target_vol_annual")
VOL_LAMBDA    = _get(gcfg, "vol_decay_lambda", 0.94)
LEV_CAP       = _get(gcfg, "vol_cap_multiplier", 2.0, "leverage_cap_multiplier")
TC_BP         = 2.0

# Sector risk budgets (commodity-focused)
COMMODITY_RISK_BUDGET    = 0.70
DIVERSIFIER_RISK_BUDGET  = 0.30  # rates + equities + fx
PORTFOLIO_VOL_TARGET     = 0.10
PORTFOLIO_VOL_LOOKBACK   = 60    # days for rolling portfolio vol
PORTFOLIO_SCALE_BOUNDS   = (0.5, 2.0)

IS_START = "2018-01-01"
IS_END   = "2026-02-27"

print("Canonical TSMOM parameters:")
print(f"  Lookback           : {LOOKBACK_DAYS}d (12 months)")
print(f"  Rebalance          : monthly")
print(f"  Per-inst vol target: {TARGET_VOL:.0%}")
print(f"  EWMA lambda        : {VOL_LAMBDA}")
print(f"  Leverage cap       : {LEV_CAP:.1f}x")
print(f"  TC per side        : {TC_BP:.1f} bp")
print(f"  Risk budget        : {COMMODITY_RISK_BUDGET:.0%} commodities / {DIVERSIFIER_RISK_BUDGET:.0%} diversifiers")
print(f"  Portfolio vol tgt  : {PORTFOLIO_VOL_TARGET:.0%}")
print(f"  IS period          : {IS_START} to {IS_END}")

Canonical TSMOM parameters:
  Lookback           : 252d (12 months)
  Rebalance          : monthly
  Per-inst vol target: 10%
  EWMA lambda        : 0.94
  Leverage cap       : 2.0x
  TC per side        : 2.0 bp
  Risk budget        : 70% commodities / 30% diversifiers
  Portfolio vol tgt  : 10%
  IS period          : 2018-01-01 to 2026-02-27


In [8]:
CONFIG_DIR = Path("config")
# Read tickers from YAML file
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

# ── Multi-asset TSMOM universe ────────────────────────────────────
universe = {
    "commodities": [
        # Precious metals futures
        "gc_fut_front", "gc_fut_second", "gc_fut_third", "xauusd_spot",
        "si_fut_front", "si_fut_second", "si_fut_third", "xagusd_spot",
        "pl_fut_front", "pl_fut_second", "xptusd_spot",
        "pa_fut_front", "pa_fut_second", "xpdusd_spot",
        # LBMA reference
        "lbma_gold_am", "lbma_gold_pm", "lbma_silver_fix",
        "lbma_platinum_am", "lbma_palladium_am",
        # Regional metals
        "shfe_au_front", "shfe_au_second", "tocom_au_front",
        # Base metals
        "hg_fut_front", "hg_fut_second",
        "lme_copper_3m", "lme_zinc_3m", "lme_nickel_3m",
        "lme_aluminium_3m", "lme_lead_3m", "lme_tin_3m", "lme_cobalt_3m",
        # Energy
        "cl_fut_front", "cl_fut_second",
        "brent_fut_front", "brent_fut_second",
        "ho_fut_front", "rb_fut_front", "gasoil_fut_front",
        "ng_fut_front", "ng_fut_second",
        "ttf_gas_front", "carbon_eua_front",
        # Agriculture
        "w_fut_front", "kw_fut_front", "c_fut_front", "s_fut_front",
        "bo_fut_front", "sm_fut_front", "ct_fut_front", "cc_fut_front",
        "kc_fut_front", "sb_fut_front", "lc_fut_front", "fc_fut_front",
        "lh_fut_front", "oz_fut_front", "rr_fut_front", "lumber_fut_front",
    ],
    "rates": [
        # SOFR OIS — full 15-pillar curve
        "sofr_ois_1w", "sofr_ois_2w", "sofr_ois_1m", "sofr_ois_2m", "sofr_ois_3m",
        "sofr_ois_4m", "sofr_ois_5m", "sofr_ois_6m", "sofr_ois_7m", "sofr_ois_8m",
        "sofr_ois_9m", "sofr_ois_10m", "sofr_ois_11m", "sofr_ois_1y", "sofr_ois_2y",
        # SOFR futures strip
        "sofr_fut_front", "sofr_fut_second", "sofr_fut_third", "sofr_fut_fourth",
        # UST futures
        "ust_2y_fut", "ust_5y_fut", "ust_10y_fut", "ust_30y_fut",
        # XAU swaps — full 13-tenor grid
        "xau_swap_1m", "xau_swap_2m", "xau_swap_3m", "xau_swap_4m", "xau_swap_5m",
        "xau_swap_6m", "xau_swap_7m", "xau_swap_8m", "xau_swap_9m", "xau_swap_10m",
        "xau_swap_11m", "xau_swap_1y", "xau_swap_2y",
        # XAG swaps — full 13-tenor grid
        "xag_swap_1m", "xag_swap_2m", "xag_swap_3m", "xag_swap_4m", "xag_swap_5m",
        "xag_swap_6m", "xag_swap_7m", "xag_swap_8m", "xag_swap_9m", "xag_swap_10m",
        "xag_swap_11m", "xag_swap_1y", "xag_swap_2y",
        # Fed Funds / US yields
        "fed_funds_target",
        "us_2y_yield", "us_5y_yield", "us_10y_yield", "us_30y_yield",
        # TIPS / Breakevens
        "tips_5y_real_yield", "tips_10y_real_yield", "breakeven_5y", "breakeven_10y",
        # Global OIS
        "tonar_ois_1y", "estr_ois_1y", "sonia_ois_1y", "corra_ois_1y", "aonia_ois_1y",
        # Short-end money markets
        "euribor_3m_fut_front", "sonia_fut_front", "nibor_3m", "stibor_3m",
        # Government bond futures
        "bund_fut_front", "bobl_fut_front", "schatz_fut_front",
        "oat_fut_front", "btp_fut_front", "gilt_fut_front",
        "jgb_fut_front", "aus_bond_fut_front", "cad_bond_fut_front",
        # Credit spreads
        "us_ig_spread", "us_hy_spread", "itraxx_europe_5y",
        # Inflation linkers
        "uk_rii_10y", "fr_oati_10y",
    ],
    "equities": [
        # US
        "es_fut_front", "nq_fut_front", "russell2000_fut",
        # Europe
        "stoxx50_fut_front", "dax_fut_front", "ftse_fut_front",
        "cac_fut_front", "smi_fut_front", "ibex_fut_front",
        # Asia-Pacific
        "nikkei_fut_front", "hsi_fut_front", "asx_fut_front",
        "kospi_fut_front",
        # Americas
        "tsx_fut_front", "omx_fut_front",
    ],
    "fx": [
        # G10 spot
        "dxy_index", "eurusd_spot", "usdjpy_spot", "gbpusd_spot", "audusd_spot",
        "usdchf_spot", "usdcnh_spot", "usdcny_spot", "nzdusd_spot", "usdcad_spot",
        # EM and commodity FX spot
        "usdnok_spot", "usdsek_spot", "usdsgd_spot", "usdbrl_spot", "usdmxn_spot",
        "usdkrw_spot", "usdinr_spot", "usdtry_spot", "usdzar_spot", "usdpln_spot",
        "usdclp_spot", "usdidr_spot",
        # FX forwards (existing)
        "eurusd_3m_fwd", "usdjpy_3m_fwd", "gbpusd_3m_fwd", "audusd_3m_fwd", "usdcnh_3m_fwd",
        # FX forwards (new)
        "usdcad_3m_fwd", "usdchf_3m_fwd", "usdnok_3m_fwd",
        "usdsek_3m_fwd", "usdmxn_3m_fwd", "usdsgd_3m_fwd",
    ],
    "risk_indicators": [
        "vix", "move_index", "gold_vol_index",
        "silver_vol_index", "oil_vol_index", "vstoxx_index",
        "vxn_index", "rvx_index", "skew_index", "vxeem_index",
        "ted_spread",
    ],
}

# Human-readable labels
LABELS = {
    # ── Precious metals ──────────────────────────────────────────────────
    "gc_fut_front":       "Gold Front (GC1)",
    "gc_fut_second":      "Gold 2nd (GC2)",
    "gc_fut_third":       "Gold 3rd (GC3)",
    "xauusd_spot":        "Gold Spot (XAU)",
    "si_fut_front":       "Silver Front (SI1)",
    "si_fut_second":      "Silver 2nd (SI2)",
    "si_fut_third":       "Silver 3rd (SI3)",
    "xagusd_spot":        "Silver Spot (XAG)",
    "pl_fut_front":       "Platinum (PL1)",
    "pl_fut_second":      "Platinum 2nd (PL2)",
    "xptusd_spot":        "Platinum Spot (XPT)",
    "pa_fut_front":       "Palladium (PA1)",
    "pa_fut_second":      "Palladium 2nd (PA2)",
    "xpdusd_spot":        "Palladium Spot (XPD)",
    "lbma_gold_am":       "LBMA Gold AM",
    "lbma_gold_pm":       "LBMA Gold PM",
    "lbma_silver_fix":    "LBMA Silver Fix",
    "lbma_platinum_am":   "LBMA Platinum AM",
    "lbma_palladium_am":  "LBMA Palladium AM",
    # ── Regional metals ──────────────────────────────────────────────────
    "shfe_au_front":      "SHFE Gold Front",
    "shfe_au_second":     "SHFE Gold 2nd",
    "tocom_au_front":     "TOCOM Gold Front",
    # ── Base metals ──────────────────────────────────────────────────────
    "hg_fut_front":       "Copper (HG1)",
    "hg_fut_second":      "Copper 2nd (HG2)",
    "lme_copper_3m":      "LME Copper 3M",
    "lme_zinc_3m":        "LME Zinc 3M",
    "lme_nickel_3m":      "LME Nickel 3M",
    "lme_aluminium_3m":   "LME Aluminium 3M",
    "lme_lead_3m":        "LME Lead 3M",
    "lme_tin_3m":         "LME Tin 3M",
    "lme_cobalt_3m":      "LME Cobalt 3M",
    # ── Energy ───────────────────────────────────────────────────────────
    "cl_fut_front":       "WTI Crude (CL1)",
    "cl_fut_second":      "WTI Crude 2nd (CL2)",
    "brent_fut_front":    "Brent Crude (CO1)",
    "brent_fut_second":   "Brent Crude 2nd (CO2)",
    "ho_fut_front":       "Heating Oil (HO1)",
    "rb_fut_front":       "RBOB Gasoline (XB1)",
    "gasoil_fut_front":   "ICE Gasoil (QS1)",
    "ng_fut_front":       "Nat Gas (NG1)",
    "ng_fut_second":      "Nat Gas 2nd (NG2)",
    "ttf_gas_front":      "TTF Nat Gas (TTF1)",
    "carbon_eua_front":   "EU Carbon EUA",
    # ── Agriculture ──────────────────────────────────────────────────────
    "w_fut_front":        "Wheat (W1)",
    "kw_fut_front":       "KC Hard Red Wheat (KW1)",
    "c_fut_front":        "Corn (C1)",
    "s_fut_front":        "Soybeans (S1)",
    "bo_fut_front":       "Soybean Oil (BO1)",
    "sm_fut_front":       "Soybean Meal (SM1)",
    "ct_fut_front":       "Cotton #2 (CT1)",
    "cc_fut_front":       "Cocoa (CC1)",
    "kc_fut_front":       "Coffee (KC1)",
    "sb_fut_front":       "Sugar #11 (SB1)",
    "lc_fut_front":       "Live Cattle (LC1)",
    "fc_fut_front":       "Feeder Cattle (FC1)",
    "lh_fut_front":       "Lean Hogs (LH1)",
    "oz_fut_front":       "Oats (OZ1)",
    "rr_fut_front":       "Rough Rice (RR1)",
    "lumber_fut_front":   "Lumber (LB1)",
    # ── SOFR OIS ─────────────────────────────────────────────────────────
    "sofr_ois_1w":        "SOFR OIS 1W",
    "sofr_ois_2w":        "SOFR OIS 2W",
    "sofr_ois_1m":        "SOFR OIS 1M",
    "sofr_ois_2m":        "SOFR OIS 2M",
    "sofr_ois_3m":        "SOFR OIS 3M",
    "sofr_ois_4m":        "SOFR OIS 4M",
    "sofr_ois_5m":        "SOFR OIS 5M",
    "sofr_ois_6m":        "SOFR OIS 6M",
    "sofr_ois_7m":        "SOFR OIS 7M",
    "sofr_ois_8m":        "SOFR OIS 8M",
    "sofr_ois_9m":        "SOFR OIS 9M",
    "sofr_ois_10m":       "SOFR OIS 10M",
    "sofr_ois_11m":       "SOFR OIS 11M",
    "sofr_ois_1y":        "SOFR OIS 1Y",
    "sofr_ois_2y":        "SOFR OIS 2Y",
    # ── SOFR futures ─────────────────────────────────────────────────────
    "sofr_fut_front":     "SOFR Fut 1st",
    "sofr_fut_second":    "SOFR Fut 2nd",
    "sofr_fut_third":     "SOFR Fut 3rd",
    "sofr_fut_fourth":    "SOFR Fut 4th",
    # ── UST futures ──────────────────────────────────────────────────────
    "ust_2y_fut":         "UST 2Y (TU)",
    "ust_5y_fut":         "UST 5Y (FV)",
    "ust_10y_fut":        "UST 10Y (TY)",
    "ust_30y_fut":        "UST 30Y (US)",
    # ── XAU swaps ────────────────────────────────────────────────────────
    "xau_swap_1m":        "XAU 1M Swap",
    "xau_swap_2m":        "XAU 2M Swap",
    "xau_swap_3m":        "XAU 3M Swap",
    "xau_swap_4m":        "XAU 4M Swap",
    "xau_swap_5m":        "XAU 5M Swap",
    "xau_swap_6m":        "XAU 6M Swap",
    "xau_swap_7m":        "XAU 7M Swap",
    "xau_swap_8m":        "XAU 8M Swap",
    "xau_swap_9m":        "XAU 9M Swap",
    "xau_swap_10m":       "XAU 10M Swap",
    "xau_swap_11m":       "XAU 11M Swap",
    "xau_swap_1y":        "XAU 1Y Swap",
    "xau_swap_2y":        "XAU 2Y Swap",
    # ── XAG swaps ────────────────────────────────────────────────────────
    "xag_swap_1m":        "XAG 1M Swap",
    "xag_swap_2m":        "XAG 2M Swap",
    "xag_swap_3m":        "XAG 3M Swap",
    "xag_swap_4m":        "XAG 4M Swap",
    "xag_swap_5m":        "XAG 5M Swap",
    "xag_swap_6m":        "XAG 6M Swap",
    "xag_swap_7m":        "XAG 7M Swap",
    "xag_swap_8m":        "XAG 8M Swap",
    "xag_swap_9m":        "XAG 9M Swap",
    "xag_swap_10m":       "XAG 10M Swap",
    "xag_swap_11m":       "XAG 11M Swap",
    "xag_swap_1y":        "XAG 1Y Swap",
    "xag_swap_2y":        "XAG 2Y Swap",
    # ── Fed Funds / US yields ────────────────────────────────────────────
    "fed_funds_target":   "Fed Funds Target",
    "us_2y_yield":        "US 2Y Yield",
    "us_5y_yield":        "US 5Y Yield",
    "us_10y_yield":       "US 10Y Yield",
    "us_30y_yield":       "US 30Y Yield",
    # ── TIPS / Breakevens ────────────────────────────────────────────────
    "tips_5y_real_yield":  "TIPS 5Y Real",
    "tips_10y_real_yield": "TIPS 10Y Real",
    "breakeven_5y":        "5Y Breakeven",
    "breakeven_10y":       "10Y Breakeven",
    # ── Global OIS ───────────────────────────────────────────────────────
    "tonar_ois_1y":       "TONAR OIS 1Y",
    "estr_ois_1y":        "ESTR OIS 1Y",
    "sonia_ois_1y":       "SONIA OIS 1Y",
    "corra_ois_1y":       "CORRA OIS 1Y",
    "aonia_ois_1y":       "AONIA OIS 1Y",
    # ── Short-end money markets ──────────────────────────────────────────
    "euribor_3m_fut_front": "Euribor 3M (ER1)",
    "sonia_fut_front":      "SONIA Fut (SQ1)",
    "nibor_3m":             "NIBOR 3M",
    "stibor_3m":            "STIBOR 3M",
    # ── Government bond futures ──────────────────────────────────────────
    "bund_fut_front":     "German Bund (RX1)",
    "bobl_fut_front":     "German Bobl (OE1)",
    "schatz_fut_front":   "German Schatz (DU1)",
    "oat_fut_front":      "French OAT (OAT1)",
    "btp_fut_front":      "Italian BTP (IK1)",
    "gilt_fut_front":     "UK Gilt (G 1)",
    "jgb_fut_front":      "Japan JGB (JB1)",
    "aus_bond_fut_front": "Aus 10Y Bond (YM1)",
    "cad_bond_fut_front": "Canada 10Y Bond (CN1)",
    # ── Credit spreads ───────────────────────────────────────────────────
    "us_ig_spread":       "US IG OAS",
    "us_hy_spread":       "US HY OAS",
    "itraxx_europe_5y":   "iTraxx Europe 5Y",
    # ── Inflation linkers ────────────────────────────────────────────────
    "uk_rii_10y":         "UK 10Y RPI Yield",
    "fr_oati_10y":        "France 10Y OATI Yield",
    # ── FX carry forwards ────────────────────────────────────────────────
    "eurusd_3m_fwd":      "EURUSD 3M Fwd",
    "usdjpy_3m_fwd":      "USDJPY 3M Fwd",
    "gbpusd_3m_fwd":      "GBPUSD 3M Fwd",
    "audusd_3m_fwd":      "AUDUSD 3M Fwd",
    "usdcnh_3m_fwd":      "USDCNH 3M Fwd",
    "usdcad_3m_fwd":      "USDCAD 3M Fwd",
    "usdchf_3m_fwd":      "USDCHF 3M Fwd",
    "usdnok_3m_fwd":      "USDNOK 3M Fwd",
    "usdsek_3m_fwd":      "USDSEK 3M Fwd",
    "usdmxn_3m_fwd":      "USDMXN 3M Fwd",
    "usdsgd_3m_fwd":      "USDSGD 3M Fwd",
    # ── Equities ─────────────────────────────────────────────────────────
    "es_fut_front":       "S&P 500 (ES1)",
    "nq_fut_front":       "Nasdaq (NQ1)",
    "russell2000_fut":    "Russell 2000 (RTY1)",
    "stoxx50_fut_front":  "EuroStoxx 50 (VG1)",
    "dax_fut_front":      "DAX (GX1)",
    "ftse_fut_front":     "FTSE 100 (Z 1)",
    "cac_fut_front":      "CAC 40 (CF1)",
    "smi_fut_front":      "Swiss SMI",
    "ibex_fut_front":     "IBEX 35 (IB1)",
    "nikkei_fut_front":   "Nikkei 225 (NK1)",
    "hsi_fut_front":      "Hang Seng (HI1)",
    "asx_fut_front":      "ASX 200 (XP1)",
    "kospi_fut_front":    "KOSPI 200 (KM1)",
    "tsx_fut_front":      "TSX 60 (PT1)",
    "omx_fut_front":      "OMX Stockholm (QC1)",
    # ── FX spot ──────────────────────────────────────────────────────────
    "dxy_index":          "DXY Index",
    "eurusd_spot":        "EURUSD",
    "usdjpy_spot":        "USDJPY",
    "gbpusd_spot":        "GBPUSD",
    "audusd_spot":        "AUDUSD",
    "usdchf_spot":        "USDCHF",
    "usdcnh_spot":        "USDCNH",
    "usdcny_spot":        "USDCNY",
    "nzdusd_spot":        "NZDUSD",
    "usdcad_spot":        "USDCAD",
    "usdnok_spot":        "USDNOK",
    "usdsek_spot":        "USDSEK",
    "usdsgd_spot":        "USDSGD",
    "usdbrl_spot":        "USDBRL",
    "usdmxn_spot":        "USDMXN",
    "usdkrw_spot":        "USDKRW",
    "usdinr_spot":        "USDINR",
    "usdtry_spot":        "USDTRY",
    "usdzar_spot":        "USDZAR",
    "usdpln_spot":        "USDPLN",
    "usdclp_spot":        "USDCLP",
    "usdidr_spot":        "USDIDR",
    # ── Risk indicators ───────────────────────────────────────────────────
    "vix":                "VIX",
    "move_index":         "MOVE Index",
    "gold_vol_index":     "Gold Vol (GVZ)",
    "silver_vol_index":   "Silver Vol (XAGVOL)",
    "oil_vol_index":      "Oil Vol (OVX)",
    "vstoxx_index":       "VSTOXX (V2X)",
    "vxn_index":          "Nasdaq VIX (VXN)",
    "rvx_index":          "Russell 2000 VIX (RVX)",
    "skew_index":         "SKEW Index",
    "vxeem_index":        "EM Equity Vol (VXEEM)",
    "ted_spread":         "TED Spread",
}

# Map instrument -> bucket for risk budgeting
# commodities → COMMODITY_RISK_BUDGET (70%)
# rates / equities / fx / risk_indicators → DIVERSIFIER_RISK_BUDGET (30%)
INST_BUCKET = {}
for bucket, instruments in universe.items():
    for inst in instruments:
        INST_BUCKET[inst] = bucket

# ── Resolve tickers from YAML ────────────────────────────────────
def resolve_ticker(logical_name: str, ticker_map: dict) -> str:
    """Resolve logical name to Bloomberg ticker; return None if missing."""
    for group in ticker_map.values():
        if isinstance(group, dict) and logical_name in group:
            return group[logical_name]
    return None

resolved = {}
missing = []
for bucket, instruments in universe.items():
    for inst in instruments:
        bbg = resolve_ticker(inst, tickers)
        if bbg:
            resolved[inst] = bbg
        else:
            missing.append(inst)

n_total = sum(len(v) for v in universe.values())
print(f"Universe: {n_total} instruments across {len(universe)} buckets")
print(f"Resolved: {len(resolved)} | Missing: {len(missing)}")
if missing:
    print(f"  Missing tickers (will be skipped): {missing}")

print("\nBucket breakdown:")
for bucket, instruments in universe.items():
    n_resolved = sum(1 for i in instruments if i in resolved)
    print(f"  {bucket:18s}: {n_resolved}/{len(instruments)} resolved")

Universe: 97 instruments across 5 buckets
Resolved: 97 | Missing: 0

Bucket breakdown:
  commodities    : 29/29 resolved
  rates          : 46/46 resolved
  equities       : 4/4 resolved
  fx             : 15/15 resolved
  risk_indicators: 3/3 resolved


In [ ]:
# ════════════════════════════════════════════════════════════════
# Physical market & positioning additions
# ────────────────────────────────────────────────────────────────
# Tier-1 missing data identified from gold RV research:
#   1. COMEX warehouse stocks (GC + SI)  — weekly CME report
#      Physical inventory levels drive lease rate dynamics.
#      When stocks low → spread tightens / backwardation risk.
#   2. CFTC COT Managed Money (GC + SI)  — weekly CFTC release
#      Extreme spec positioning precedes spread richening/cheapening.
#   3. ETF price proxies (GLD, SLV)      — daily
#      Relative change vs spot reveals creation/redemption flows,
#      which move the physical float and lease rates.
#
# Bloomberg tickers are hardcoded here (no tickers.yaml entry needed).
# All series use PX_LAST field and are compatible with BQuantDataLoader.
# ════════════════════════════════════════════════════════════════

NEW_INSTRUMENTS = {
    # ── COMEX Warehouse Stocks ────────────────────────────────────
    # Weekly CME report; PX_LAST = troy oz held at COMEX vaults
    "comex_gold_registered":   ("COMXGDRG Index",  "commodities"),
    "comex_gold_eligible":     ("COMXGDEL Index",  "commodities"),
    "comex_gold_total":        ("COMXGDTT Index",  "commodities"),
    "comex_silver_registered": ("COMXSIRG Index",  "commodities"),
    "comex_silver_eligible":   ("COMXSIEL Index",  "commodities"),
    "comex_silver_total":      ("COMXSITT Index",  "commodities"),
    # ── CFTC Commitment of Traders — Managed Money ───────────────
    # Weekly CFTC legacy disaggregated; PX_LAST = number of contracts
    "cot_gc_mm_long":          ("CFNAGCML Index",  "positioning"),
    "cot_gc_mm_short":         ("CFNAGCMS Index",  "positioning"),
    "cot_gc_mm_net":           ("CFNAGCMN Index",  "positioning"),
    "cot_si_mm_long":          ("CFNASIML Index",  "positioning"),
    "cot_si_mm_short":         ("CFNASIMS Index",  "positioning"),
    "cot_si_mm_net":           ("CFNASIMN Index",  "positioning"),
    # ── ETF Price Proxies ─────────────────────────────────────────
    # Daily; pct-change relative to spot reveals fund creation/redemption
    "gld_etf_price":           ("GLD US Equity",   "risk_indicators"),
    "slv_etf_price":           ("SLV US Equity",   "risk_indicators"),
}

NEW_LABELS = {
    "comex_gold_registered":   "COMEX Gold Registered (oz)",
    "comex_gold_eligible":     "COMEX Gold Eligible (oz)",
    "comex_gold_total":        "COMEX Gold Total (oz)",
    "comex_silver_registered": "COMEX Silver Registered (oz)",
    "comex_silver_eligible":   "COMEX Silver Eligible (oz)",
    "comex_silver_total":      "COMEX Silver Total (oz)",
    "cot_gc_mm_long":          "COT Gold MM Long",
    "cot_gc_mm_short":         "COT Gold MM Short",
    "cot_gc_mm_net":           "COT Gold MM Net",
    "cot_si_mm_long":          "COT Silver MM Long",
    "cot_si_mm_short":         "COT Silver MM Short",
    "cot_si_mm_net":           "COT Silver MM Net",
    "gld_etf_price":           "GLD ETF Price",
    "slv_etf_price":           "SLV ETF Price",
}

# ── Register in universe, LABELS, INST_BUCKET, resolved ──────────
universe.setdefault("positioning", [])

for inst, (bbg_ticker, bucket) in NEW_INSTRUMENTS.items():
    # Add to universe bucket list (avoid duplicates on re-run)
    if inst not in universe.get(bucket, []):
        universe.setdefault(bucket, []).append(inst)
    # Register label, bucket mapping, and resolved ticker
    LABELS[inst]      = NEW_LABELS[inst]
    INST_BUCKET[inst] = bucket
    resolved[inst]    = bbg_ticker
    if inst in missing:
        missing.remove(inst)

n_total = sum(len(v) for v in universe.values())
print(f"Updated universe : {n_total} instruments ({len(resolved)} resolved, {len(missing)} missing)")
print(f"  +{len(NEW_INSTRUMENTS)} new  →  COMEX warehouse stocks (6) | CFTC COT MM (6) | ETF proxies (2)")
print()
print(f"  {'Instrument':30s}  {'Bloomberg Ticker':22s}  {'Bucket'}")
print("  " + "-" * 70)
for inst, (bbg, bucket) in NEW_INSTRUMENTS.items():
    print(f"  {inst:30s}  {bbg:22s}  {bucket}")

In [9]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL."""

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def resolve(self, logical_name: str) -> str:
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        raise KeyError(f"{logical_name} not in tickers.yaml")

    def get_history(self, logical_name: str, start: str, end: str,
                    field: str = "PX_LAST") -> pd.Series:
        bbg = self.resolve(logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                return pd.Series(dtype=float, name=logical_name)
            df_fixed = df.set_index('DATE')
            series = df_fixed[field]
            series.index = pd.to_datetime(series.index, errors='coerce')
            series = series.dropna()
            series = series[~series.index.duplicated(keep='last')]
            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"
            if not isinstance(series.index, pd.DatetimeIndex):
                series.index = pd.to_datetime(series.index)
            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)


# ── Fetch all resolved instruments ────────────────────────────────
loader = BQuantDataLoader(tickers)
prices_raw = {}
fetch_status = []

for inst in resolved:
    label = LABELS.get(inst, inst)
    bucket = INST_BUCKET[inst]
    print(f"  {label:22s}", end=" ")
    s = loader.get_history(inst, IS_START, IS_END)
    n_obs = len(s)
    if n_obs > 0:
        prices_raw[inst] = s
        print(f"OK  {n_obs:>5d} obs  [{s.index[0]:%Y-%m-%d} -> {s.index[-1]:%Y-%m-%d}]")
    else:
        print("MISSING")
    fetch_status.append({"instrument": inst, "label": label, "bucket": bucket,
                         "obs": n_obs, "status": "OK" if n_obs > 0 else "MISSING"})

# Build aligned panel
prices_df = pd.DataFrame(prices_raw).sort_index()
prices_df = prices_df[prices_df.index.notna()].ffill()

# Drop instruments with insufficient history
MIN_OBS = LOOKBACK_DAYS + 30
sufficient = prices_df.count() >= MIN_OBS
dropped_insts = prices_df.columns[~sufficient].tolist()
if dropped_insts:
    print(f"\nDropped (< {MIN_OBS} obs): {[LABELS.get(i,i) for i in dropped_insts]}")
prices_df = prices_df[prices_df.columns[sufficient]]

# Update INST_BUCKET to only include surviving instruments
active_instruments = list(prices_df.columns)
active_buckets = {}
for inst in active_instruments:
    b = INST_BUCKET.get(inst)
    if b:
        active_buckets.setdefault(b, []).append(inst)

print(f"\nFinal panel: {prices_df.shape[0]} days x {prices_df.shape[1]} instruments")

# ── Breadth report ────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  BREADTH REPORT")
print("=" * 65)
status_df = pd.DataFrame(fetch_status)
for bucket in universe:
    bdf = status_df[status_df["bucket"] == bucket]
    n_total = len(bdf)
    n_ok = (bdf["status"] == "OK").sum()
    n_sufficient = sum(1 for i in universe[bucket] if i in prices_df.columns)
    print(f"  {bucket:15s}: {n_total} defined | {n_ok} fetched | {n_sufficient} with >= {MIN_OBS}d history")

# Missing data counts per instrument
miss_counts = prices_df.isna().sum()
if miss_counts.sum() > 0:
    print("\nMissing data counts (after ffill):")
    for inst in prices_df.columns:
        m = miss_counts[inst]
        if m > 0:
            print(f"  {LABELS.get(inst, inst):22s}: {m}")
else:
    print("\nNo missing data after forward-fill.")

prices_df.tail(3)

  Gold Front (GC1)       OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Gold 2nd (GC2)         OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Gold 3rd (GC3)         OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Gold Spot (XAU)        OK   2117 obs  [2018-01-02 -> 2026-02-27]
  Silver Front (SI1)     OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Silver 2nd (SI2)       OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Silver 3rd (SI3)       OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Silver Spot (XAG)      OK   2126 obs  [2018-01-02 -> 2026-02-27]
  Platinum (PL1)         OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Platinum Spot (XPT)    OK   2131 obs  [2018-01-02 -> 2026-02-27]
  Palladium (PA1)        OK   2054 obs  [2018-01-02 -> 2026-02-27]
  Palladium Spot (XPD)   OK   2126 obs  [2018-01-02 -> 2026-02-27]
  Copper (HG1)           OK   2054 obs  [2018-01-02 -> 2026-02-27]
  LME Copper 3M          OK   2063 obs  [2018-01-02 -> 2026-02-27]
  LME Zinc 3M            OK   2063 obs  [2018-01-02 -> 2026-02

,gc_fut_front,gc_fut_second,gc_fut_third,xauusd_spot,si_fut_front,si_fut_second,si_fut_third,xagusd_spot,pl_fut_front,xptusd_spot,...,nzdusd_spot,usdcad_spot,eurusd_3m_fwd,usdjpy_3m_fwd,gbpusd_3m_fwd,audusd_3m_fwd,usdcnh_3m_fwd,vix,move_index,gold_vol_index
date,,,,,,,,,,,,,,,,,,,,,
2026-02-25,5226.2,5266.1,5305.4,5164.78,90.988,91.630,92.280,89.2296,2331.7,2289.10,...,0.5999,1.3676,51.15,-119.50,3.59,-4.35,-691.99,17.93,62.43,34.96
2026-02-26,5194.2,5233.7,5272.8,5184.97,86.998,87.584,88.221,88.3000,2239.6,2284.02,...,0.5977,1.3681,51.57,-120.05,4.05,-4.60,-842.99,18.63,63.93,33.07
2026-02-27,5247.9,5287.6,5327.1,5278.93,92.682,93.291,93.969,93.7867,2373.5,2369.00,...,0.5998,1.3640,51.65,-119.90,4.22,-4.65,-784.51,19.86,73.38,33.23


In [ ]:
# ════════════════════════════════════════════════════════════════
# Contract dates: FDD / FND / LTD / OI
# Fetches delivery and notice dates + open interest for all key
# futures contracts. Saved to data_contract_dates.csv (separate
# from price data to keep data.csv clean).
# ════════════════════════════════════════════════════════════════

# Contracts: (bbg_ticker, metal_key, position_label)
_CONTRACTS = [
    ("GC1 Comdty", "gc", "front"),
    ("GC2 Comdty", "gc", "second"),
    ("SI1 Comdty", "si", "front"),
    ("SI2 Comdty", "si", "second"),
    ("PL1 Comdty", "pl", "front"),
    ("PA1 Comdty", "pa", "front"),
    ("HG1 Comdty", "hg", "front"),
    ("HG2 Comdty", "hg", "second"),
    ("CL1 Comdty", "cl", "front"),
    ("CL2 Comdty", "cl", "second"),
    ("CO1 Comdty", "co", "front"),
    ("NG1 Comdty", "ng", "front"),
    ("NG2 Comdty", "ng", "second"),
]

# BQL date fields  (pulled as daily time series via dates=range)
_DATE_FIELD_NAMES = ["fdd", "fnd", "ltd"]

cd_raw: dict = {}

print("Fetching contract dates + OI ...")
for bbg, metal, pos in _CONTRACTS:
    tag = f"{metal}_{pos}"
    print(f"  {tag:12s} ({bbg:12s})", end="  ")
    try:
        req = bql.Request(bbg, {
            "fdd": bq.data.fut_dlv_dt_first(dates=bq.func.range(IS_START, IS_END)),
            "fnd": bq.data.fut_notice_first(dates=bq.func.range(IS_START, IS_END)),
            "ltd": bq.data.last_tradeable_dt(dates=bq.func.range(IS_START, IS_END)),
            "oi":  bq.data.open_int(dates=bq.func.range(IS_START, IS_END)),
        })
        resp = bq.execute(req)

        for idx, fname in enumerate(["fdd", "fnd", "ltd", "oi"]):
            col = f"{metal}_{fname}_{pos}"
            try:
                df_i = resp[idx].df()
                if df_i.empty:
                    print(f"{fname}:MISS ", end="")
                    continue
                df_fixed = df_i.set_index("DATE")
                s = df_fixed.iloc[:, 0].copy()
                s.index = pd.to_datetime(s.index, errors="coerce")
                s = s[~s.index.duplicated(keep="last")].sort_index()
                s.name = col
                s.index.name = "date"
                if fname in ("fdd", "fnd", "ltd"):
                    s = pd.to_datetime(s, errors="coerce")
                else:
                    s = pd.to_numeric(s, errors="coerce")
                cd_raw[col] = s
                print(f"{fname}:OK ", end="")
            except Exception as e_inner:
                print(f"{fname}:ERR ", end="")
        print()
    except Exception as exc:
        print(f"ERROR: {exc}")

# Build aligned panel
cd_df = pd.DataFrame(cd_raw).sort_index()
cd_df = cd_df[cd_df.index.notna()].ffill()

# DTE columns: calendar days from observation date to FDD (front month only)
for metal in ("gc", "si", "pl", "pa", "hg", "cl", "co", "ng"):
    fdd_col = f"{metal}_fdd_front"
    dte_col = f"{metal}_dte_fdd"
    if fdd_col in cd_df.columns:
        fdd_dates = pd.to_datetime(cd_df[fdd_col], errors="coerce")
        cd_df[dte_col] = (fdd_dates - cd_df.index.to_series()).dt.days

# Save
cd_df.to_csv("data_contract_dates.csv")
print(f"
Contract dates saved: {cd_df.shape[0]} rows x {cd_df.shape[1]} columns -> data_contract_dates.csv")

# Validation table (10 recent rows, GC columns)
val_cols = [c for c in ["gc_fdd_front", "gc_fnd_front", "gc_ltd_front", "gc_dte_fdd", "gc_oi_front"] if c in cd_df.columns]
if val_cols:
    print(f"
{'Date':12s}", "  ".join(f"{c:20s}" for c in val_cols))
    recent = cd_df[val_cols].dropna(how="all").tail(10)
    for dt, row in recent.iterrows():
        vals = "  ".join(f"{str(row[c])[:20]:20s}" for c in val_cols)
        print(f"{str(dt)[:10]:12s}  {vals}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Export: data.csv  (prices/rates/vols/positioning — no date columns)
# Column order: commodities → rates → equities → fx → risk_indicators → positioning
# data_contract_dates.csv already saved in the cell above.
# ════════════════════════════════════════════════════════════════

PREV_COLS = 97  # previous data.csv column count (97 after last run)

# Build ordered column list by bucket
ordered_cols = []
for bucket in ("commodities", "rates", "equities", "fx", "risk_indicators", "positioning"):
    for inst in universe.get(bucket, []):
        if inst in prices_df.columns:
            ordered_cols.append(inst)

# Ensure all float64
export_df = prices_df[ordered_cols].copy()
for c in export_df.columns:
    export_df[c] = pd.to_numeric(export_df[c], errors="coerce")

export_df.to_csv("data.csv")

# ── Breadth report ────────────────────────────────────────────────
status_df = pd.DataFrame(fetch_status)
n_new = len(export_df.columns)

print(f"Previous data.csv : {PREV_COLS:>4d} columns")
print(f"New data.csv      : {n_new:>4d} columns  (+{n_new - PREV_COLS:d} added)")
print()

OLD_COUNTS = {
    "commodities":    29,
    "rates":          46,
    "equities":        4,
    "fx":             15,
    "risk_indicators": 3,
    "positioning":     0,
}

print("Bucket breakdown:")
for bucket in ("commodities", "rates", "equities", "fx", "risk_indicators", "positioning"):
    bucket_df = status_df[status_df["bucket"] == bucket] if "bucket" in status_df.columns else pd.DataFrame()
    n_total   = len(universe.get(bucket, []))
    n_ok      = int((bucket_df["status"] == "OK").sum()) if not bucket_df.empty else 0
    # positioning instruments are added via NEW_INSTRUMENTS (not in fetch_status) — count from prices_df
    if bucket == "positioning":
        n_ok = sum(1 for i in universe.get("positioning", []) if i in prices_df.columns)
    n_miss    = n_total - n_ok
    old       = OLD_COUNTS.get(bucket, 0)
    n_added   = n_total - old
    miss_list = bucket_df[bucket_df["status"] == "MISSING"]["instrument"].tolist() if not bucket_df.empty else []
    # Also flag positioning instruments not in prices_df
    if bucket == "positioning":
        miss_list = [i for i in universe.get("positioning", []) if i not in prices_df.columns]
    print(f"  {bucket:18s}: was {old:2d} -> now {n_total:3d}  (+{n_added:2d} new, {n_miss:2d} MISSING)")
    if miss_list:
        print(f"    MISSING: {', '.join(miss_list)}")

print(f"\ndata.csv saved: {export_df.shape[0]} rows x {export_df.shape[1]} columns")